In [ ]:
# ============================================================
# DOWNLOAD COCO DATASET IN GOOGLE COLAB
# ============================================================
# Colab has fast internet - this should take 15-30 minutes
# ============================================================

print("=" * 60)
print("📥 DOWNLOADING COCO DATASET IN GOOGLE COLAB")
print("=" * 60)

# Create directory
!mkdir -p coco_data

# Download annotations (~252 MB)
print("\n📥 Downloading annotations...")
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P coco_data/
print("✅ Annotations downloaded!")

# Download training images (~18 GB)
print("\n📥 Downloading training images (~18 GB)...")
print("⏳ This will take 10-20 minutes on Colab...")
!wget http://images.cocodataset.org/zips/train2017.zip -P coco_data/
print("✅ Training images downloaded!")

# Download validation images (~1 GB)
print("\n📥 Downloading validation images (~1 GB)...")
!wget http://images.cocodataset.org/zips/val2017.zip -P coco_data/
print("✅ Validation images downloaded!")

# Extract all files
print("\n📦 Extracting files...")
!unzip -q coco_data/annotations_trainval2017.zip -d coco_data/
print("✅ Annotations extracted!")

!unzip -q coco_data/train2017.zip -d coco_data/
print("✅ Training images extracted!")

!unzip -q coco_data/val2017.zip -d coco_data/
print("✅ Validation images extracted!")

# Verify
import os
train_count = len(os.listdir('coco_data/train2017'))
val_count = len(os.listdir('coco_data/val2017'))

print("\n" + "=" * 60)
print("🎉 COCO DATASET READY IN COLAB!")
print("=" * 60)
print(f"Training images: {train_count:,}")
print(f"Validation images: {val_count:,}")
print("\nNow we'll extract features and save to Google Drive...")

📥 DOWNLOADING COCO DATASET IN GOOGLE COLAB

📥 Downloading annotations...
--2026-02-06 05:20:46--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.216.62.33, 52.216.206.211, 52.217.206.73, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.216.62.33|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘coco_data/annotations_trainval2017.zip’

annotations_trainva 100%[===================>] 241.19M  59.3MB/s    in 4.5s    

2026-02-06 05:20:51 (53.6 MB/s) - ‘coco_data/annotations_trainval2017.zip’ saved [252907541/252907541]

✅ Annotations downloaded!

📥 Downloading training images (~18 GB)...
⏳ This will take 10-20 minutes on Colab...
--2026-02-06 05:20:51--  http://images.cocodataset.org/zips/train2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.202.195, 52.217.166.65, 16.15.188.176, ...
Conn

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
# We'll save extracted features to your Google Drive
# so you can download them to your local machine
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")
print("Features will be saved to: /content/drive/MyDrive/coco_features/")

# Create directory
!mkdir -p /content/drive/MyDrive/coco_features/

Mounted at /content/drive
✅ Google Drive mounted!
Features will be saved to: /content/drive/MyDrive/coco_features/


In [ ]:
# ============================================================
# EXTRACT IMAGE FEATURES USING COLAB GPU
# ============================================================
# Colab has free GPU - this will be MUCH faster than your CPU
# ============================================================

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pickle
from tqdm import tqdm
import os

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("✅ GPU available - feature extraction will be FAST!")
else:
    print("⚠️  No GPU - click Runtime → Change runtime type → GPU")

# Load ResNet50
resnet = models.resnet50(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device)
resnet.eval()

print("✅ ResNet50 loaded!")

# Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

def extract_features(image_path):
    """Extract ResNet50 features"""
    try:
        image = Image.open(image_path).convert('RGB')
        image = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            features = resnet(image)
            features = features.view(features.size(0), -1)

        return features.cpu().numpy()[0]
    except:
        return None

# Extract training features (use first 20k for faster training)
print("\n📊 Extracting training features...")
print("Using first 20,000 images (plenty for training)")

train_features = {}
train_images = sorted(os.listdir('coco_data/train2017'))[:20000]

for img_name in tqdm(train_images):
    img_path = f'coco_data/train2017/{img_name}'
    img_id = int(img_name.split('.')[0])

    features = extract_features(img_path)
    if features is not None:
        train_features[img_id] = features

print(f"✅ Extracted {len(train_features)} training features")

# Save to Google Drive
with open('/content/drive/MyDrive/coco_features/train_features.pkl', 'wb') as f:
    pickle.dump(train_features, f)

print("✅ Training features saved to Google Drive!")

# Extract validation features (use first 2k)
print("\n📊 Extracting validation features...")

val_features = {}
val_images = sorted(os.listdir('coco_data/val2017'))[:2000]

for img_name in tqdm(val_images):
    img_path = f'coco_data/val2017/{img_name}'
    img_id = int(img_name.split('.')[0])

    features = extract_features(img_path)
    if features is not None:
        val_features[img_id] = features

print(f"✅ Extracted {len(val_features)} validation features")

# Save to Google Drive
with open('/content/drive/MyDrive/coco_features/val_features.pkl', 'wb') as f:
    pickle.dump(val_features, f)

print("✅ Validation features saved to Google Drive!")

print("\n" + "=" * 60)
print("🎉 FEATURE EXTRACTION COMPLETE!")
print("=" * 60)
print("Features saved to your Google Drive:")
print("  - train_features.pkl")
print("  - val_features.pkl")
print("\nYou can now download these to your local machine!")

Using device: cuda
GPU: Tesla T4
✅ GPU available - feature extraction will be FAST!


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 123MB/s]


✅ ResNet50 loaded!

📊 Extracting training features...
Using first 20,000 images (plenty for training)


100%|██████████| 20000/20000 [05:17<00:00, 62.97it/s]


✅ Extracted 20000 training features
✅ Training features saved to Google Drive!

📊 Extracting validation features...


100%|██████████| 2000/2000 [00:31<00:00, 64.06it/s]


✅ Extracted 2000 validation features
✅ Validation features saved to Google Drive!

🎉 FEATURE EXTRACTION COMPLETE!
Features saved to your Google Drive:
  - train_features.pkl
  - val_features.pkl

You can now download these to your local machine!


In [ ]:
# ============================================================
# BUILD VOCABULARY
# ============================================================

!pip install pycocotools nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from pycocotools.coco import COCO
from nltk.tokenize import word_tokenize
from collections import Counter
import pickle

# Load COCO captions
coco = COCO('coco_data/annotations/captions_train2017.json')

# Get all captions
all_captions = []
for ann_id in coco.anns:
    caption = coco.anns[ann_id]['caption']
    all_captions.append(caption.lower())

print(f"Total captions: {len(all_captions):,}")

# Build vocabulary
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, captions):
        word_freq = Counter()
        idx = 4

        print("Tokenizing captions...")
        for caption in captions:
            tokens = word_tokenize(caption)
            word_freq.update(tokens)

        print(f"Unique tokens: {len(word_freq)}")

        for word, freq in word_freq.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

        print(f"Vocabulary size: {len(self.itos)}")

    def numericalize(self, caption):
        tokens = word_tokenize(caption.lower())
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokens]

# Build and save vocabulary
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(all_captions)

with open('/content/drive/MyDrive/coco_features/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

print("✅ Vocabulary saved to Google Drive!")
print(f"   Size: {len(vocab)} words")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


loading annotations into memory...
Done (t=0.95s)
creating index...
index created!
Total captions: 591,753
Tokenizing captions...
Unique tokens: 29286
Vocabulary size: 10322
✅ Vocabulary saved to Google Drive!
   Size: 10322 words
